In [100]:
import pandas as pd
import numpy as np
df = pd.read_csv('recipes_normalized.csv')
display(df)
np.set_printoptions(threshold=np.inf)
all_ingredients = unique_ingredients = (
    df['NER']
    .str.split(',')
    .explode()
    .str.strip()
    .str.lower()
    .dropna()
    .unique()
)


with open("output.txt", "w") as f:
    for x in all_ingredients:
        f.write(str(x) + "\n")

len(all_ingredients)


,title,ingredients,directions,link,source,NER
0,No-Bake Nut Cookies,200g firmly packed brown sugar | 120ml evapora...,"In a heavy 2-quart saucepan, mix brown sugar, ...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"brown sugar, milk, vanilla, nuts, butter, bite..."
1,Jewell Ball'S Chicken,"1 small jar chipped beef, cut up | 4 boned chi...",Place chipped beef on bottom of baking dish. |...,www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"beef, chicken breasts, cream of mushroom soup,..."
2,Creamy Corn,"2 pkg. frozen corn | 1 pkg. cream cheese, cu...","In a slow cooker, combine all ingredients. Cov...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"frozen corn, cream cheese, butter, garlic powd..."
3,Chicken Funny,1 large whole chicken | 2 cans chicken gravy ...,Boil and debone chicken. | Put bite size piece...,www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"chicken, chicken gravy, cream of mushroom soup..."
4,Reeses Cups(Candy),200g peanut butter | 150g graham cracker crumb...,Combine first four ingredients and press in 13...,www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"peanut butter, graham cracker crumbs, butter, ..."
...,...,...,...,...,...,...
49995,Caramel Frosting,2 sticks butter | 400g sugar | 240ml milk,Let butter melt; add sugar and brown. | Add mi...,www.cookbooks.com/Recipe-Details.aspx?id=271824,Gathered,"butter, sugar, milk"
49996,Barbecued Chicken Wings,20 to 30 chicken wings | 1 bottle Kraft barbe...,"Mix sauce, brown sugar, onion and water | mixi...",www.cookbooks.com/Recipe-Details.aspx?id=285631,Gathered,"chicken, barbecue sauce, brown sugar, onion, w..."
49997,Pound Cake,"3 sticks butter, beaten | 6 eggs | 800g self-r...",Bake at 275° for 1 1/2 hours. | Put toothpick ...,www.cookbooks.com/Recipe-Details.aspx?id=1020555,Gathered,"butter, eggs, flour, confectioners sugar, lemo..."
49998,Slush,350g sugar | 480ml boiling water | 5 mashed ba...,Combine sugar and boiling water. | Stir and co...,www.cookbooks.com/Recipe-Details.aspx?id=599562,Gathered,"sugar, boiling water, bananas, orange juice, w..."


10028

SyntaxError: positional argument follows keyword argument (881483881.py, line 1)

In [110]:

import pandas as pd
import numpy as np
import re
import random
from collections import defaultdict, Counter

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


UNIT_RE = re.compile(
    r'^([\d\.\/]+)\s*'
    r'(g|kg|ml|l|tsp|tbsp|cup|oz|lb|pkg\.?|can|jar|box|clove|slice|piece|pinch|dash|stick)\b',
    re.I
)


CATEGORIES = {
    "protein_meat":   ["beef","chicken","pork","turkey","lamb","steak","bacon","sausage","ham"],
    "protein_fish":   ["salmon","tuna","shrimp","cod","tilapia","crab","fish"],
    "protein_veggie": ["tofu","lentil","chickpea","black bean","kidney bean","bean"],
    "dairy":          ["milk","butter","cheese","cream","yogurt","sour cream","cream cheese"],
    "egg":            ["egg"],
    "produce_veg":    ["onion","garlic","tomato","pepper","carrot","celery","broccoli",
                       "potato","mushroom","corn","spinach","lettuce","zucchini"],
    "produce_fruit":  ["apple","banana","lemon","lime","berry","strawberr","pineapple","orange","peach"],
    "grains":         ["flour","rice","pasta","bread","oat","cracker","noodle","tortilla","cereal"],
    "baking":         ["sugar","brown sugar","baking soda","baking powder","vanilla",
                       "chocolate chip","cocoa","honey","cornstarch"],
    "pantry_staple":  ["oil","vinegar","salt","pepper","soy sauce","ketchup","mustard",
                       "mayonnaise","broth","stock","tomato sauce","worcestershire"],
    "spice_herb":     ["cumin","paprika","oregano","basil","thyme","cinnamon","ginger",
                       "garlic powder","onion powder","parsley","chili","cayenne"],
    "nuts_seeds":     ["almond","walnut","pecan","cashew","peanut","sesame"],
    "canned_goods":   ["canned","condensed","cream of mushroom","cream of chicken"],
    "frozen":         ["frozen"],
}

def get_category(ing: str) -> str:
    for cat, keywords in CATEGORIES.items():
        if any(kw in ing for kw in keywords):
            return cat
    return "other"


ARCHETYPES = {
    "home_baker":       {"baking":0.9,"dairy":0.85,"egg":0.9,"grains":0.8,
                         "pantry_staple":0.7,"nuts_seeds":0.6,"spice_herb":0.5,
                         "produce_veg":0.5,"protein_meat":0.3,"other":0.3},
    "meat_lover":       {"protein_meat":0.95,"pantry_staple":0.85,"spice_herb":0.8,
                         "dairy":0.65,"grains":0.6,"produce_veg":0.6,"egg":0.5,
                         "canned_goods":0.5,"other":0.35},
    "vegetarian":       {"produce_veg":0.95,"produce_fruit":0.8,"protein_veggie":0.85,
                         "grains":0.85,"dairy":0.75,"spice_herb":0.8,"egg":0.7,
                         "pantry_staple":0.75,"baking":0.5,"other":0.3},
    "budget_cook":      {"grains":0.9,"canned_goods":0.85,"pantry_staple":0.8,
                         "protein_meat":0.7,"produce_veg":0.65,"dairy":0.6,
                         "egg":0.7,"baking":0.4,"other":0.25},
    "health_conscious": {"produce_veg":0.95,"produce_fruit":0.9,"protein_fish":0.7,
                         "nuts_seeds":0.8,"grains":0.75,"protein_veggie":0.8,
                         "dairy":0.5,"spice_herb":0.75,"pantry_staple":0.6,"other":0.3},
    "family_cook":      {"protein_meat":0.9,"dairy":0.9,"grains":0.9,"produce_veg":0.85,
                         "pantry_staple":0.85,"canned_goods":0.8,"frozen":0.75,
                         "egg":0.85,"baking":0.65,"spice_herb":0.7,"other":0.4},
    "casual_cook":      {"pantry_staple":0.7,"dairy":0.65,"egg":0.6,"produce_veg":0.55,
                         "grains":0.5,"protein_meat":0.45,"canned_goods":0.5,"other":0.2},
}
ARCHETYPE_WEIGHTS = np.array([0.12, 0.15, 0.13, 0.18, 0.14, 0.16, 0.12])
ARCHETYPE_WEIGHTS /= ARCHETYPE_WEIGHTS.sum()



def parse_amount(s: str):
    try:
        if '/' in s:
            n, d = s.split('/')
            return float(n) / float(d)
        return float(s)
    except:
        return None


def extract_units_from_recipes(recipes_path: str, known_ings: set) -> dict:
    recipes = pd.read_csv(recipes_path, usecols=["ingredients", "NER"])
    ing_data: dict = defaultdict(lambda: defaultdict(list))

    for _, row in recipes.iterrows():
        ner_raw = re.sub(r"[\[\]'\"]", "", str(row["NER"]))
        ner_ings = [x.strip().lower() for x in ner_raw.split(",") if x.strip()]
        lines = [x.strip() for x in str(row["ingredients"]).split("|")]

        for ing in ner_ings:
            if ing not in known_ings:
                continue
            for line in lines:
                if ing in line.lower():
                    m = UNIT_RE.match(line.strip())
                    if m:
                        amt = parse_amount(m.group(1))
                        unit = m.group(2).lower().rstrip('.')
                        if amt and 0 < amt < 5000:
                            ing_data[ing][unit].append(amt)
                    break

    result = {}
    for ing, unit_dict in ing_data.items():
        best_unit = max(unit_dict, key=lambda u: len(unit_dict[u]))
        median_amt = float(np.median(unit_dict[best_unit]))
        result[ing] = (median_amt, best_unit)

    return result


def make_pantry_amount(median_amt: float, unit: str) -> str:
    multiplier = random.uniform(2, 5)
    amt = median_amt * multiplier

    if unit in ("tsp", "tbsp", "pinch", "dash", "clove"):
        amt = round(amt, 1)
    elif unit in ("g", "ml"):
        amt = max(50, round(amt / 50) * 50)
    elif unit in ("can", "pkg", "box", "jar", "piece", "slice", "stick"):
        amt = max(1, round(amt))
    else:
        amt = round(amt, 1)

    return f"{amt} {unit}"




def add_pantry_column(
    ingredients_path: str,
    recipes_path: str,
    n_users: int = 1000,
    top_n: int = 200,
) -> pd.DataFrame:
   


    with open(ingredients_path) as f:
        known_ings = {l.strip().lower() for l in f if l.strip()}
   

    
    
    unit_map = extract_units_from_recipes(recipes_path, known_ings)
    

   
    freq = Counter({ing: len([]) for ing in unit_map})  
   
    recipes_tmp = pd.read_csv(recipes_path, usecols=["NER"])
    for ner_raw in recipes_tmp["NER"]:
        clean = re.sub(r"[\[\]'\"]", "", str(ner_raw))
        for ing in [x.strip().lower() for x in clean.split(",") if x.strip()]:
            if ing in known_ings:
                freq[ing] += 1

    candidates = [ing for ing, _ in freq.most_common(top_n)]
    

    
    cat_map = {ing: get_category(ing) for ing in candidates}

    
    archetype_names = list(ARCHETYPES.keys())
    rows = []

    for uid in range(1, n_users + 1):
        arch_name = np.random.choice(archetype_names, p=ARCHETYPE_WEIGHTS)
        arch = ARCHETYPES[arch_name]

        pantry = {}
        for ing in candidates:
            cat = cat_map[ing]
            prob = arch.get(cat, arch.get("other", 0.3))
            if random.random() < prob:
                if ing in unit_map:
                    median_amt, unit = unit_map[ing]
                    pantry[ing] = make_pantry_amount(median_amt, unit)
                else:
                    pantry[ing] = "1 pkg"

        rows.append({
            "user_id":   uid,
            "archetype": arch_name,
            "pantry":    pantry,
        })

    df = pd.DataFrame(rows).set_index("user_id")
    return df



if __name__ == "__main__":
    df = add_pantry_column(
        ingredients_path="output.txt",
        recipes_path="recipes_normalized.csv",
        n_users=1000,
        top_n=200,
    )

   

    df.to_csv("user_pantries.csv")
   

In [118]:
pantries = pd.read_csv('user_pantries.csv')

pantries.drop(columns=["archetype"])
display(pantries)

,user_id,archetype,pantry
0,1,vegetarian,"{'salt': '1.0 tsp', 'sugar': '550 g', 'flour':..."
1,2,casual_cook,"{'flour': '1700 g', 'eggs': '1150 ml', 'margar..."
2,3,family_cook,"{'salt': '2.1 tsp', 'flour': '1400 g', 'eggs':..."
3,4,health_conscious,"{'salt': '1.2 tsp', 'onion': '300 g', 'pepper'..."
4,5,meat_lover,"{'flour': '1350 g', 'milk': '1050 ml', 'water'..."
...,...,...,...
995,996,home_baker,"{'salt': '1.4 tsp', 'sugar': '700 g', 'flour':..."
996,997,casual_cook,"{'salt': '2.1 tsp', 'flour': '1800 g', 'eggs':..."
997,998,meat_lover,"{'flour': '1350 g', 'butter': '350 g', 'onion'..."
998,999,casual_cook,"{'salt': '1.2 tsp', 'flour': '1300 g', 'eggs':..."
